### USPS Model: Using proportion variables $x_{i,r}$

In [1]:
# import the csv data files using pandas
import pandas as pd

# create DataFrame for the deliveries
df_deliveries = pd.read_csv("deliveries.csv").set_index('Delivery')

# convert the DataFrame into dictionaries (for convenience--it is also possible to directly use df_deliveries)
Origin = {}
Destination = {}
Available = {}
LatestArrival = {}
Volume = {}
for i in df_deliveries.index:
    Origin[i] = df_deliveries['Origin'][i]  # this assigns the i-th element
    Destination[i] = df_deliveries['Destination'][i]
    Available[i] = df_deliveries['Available'][i]
    LatestArrival[i] = df_deliveries['LatestArrival'][i]
    Volume[i] = df_deliveries['Volume'][i]

In [2]:
# create DataFrame for the trips
df_trips = pd.read_csv("trips.csv").set_index('Trip')

# Convert the DataFrame into dictionaries (for convenience--it is also possible to directly use df_trips).
# The following parameters are easy to define:
Cost = {}
Capacity = {}
Leave = {}
Arrive = {}
for i in df_trips.index:
    Cost[i] = df_trips['Cost'][i]
    Capacity[i] = df_trips['Capacity'][i]
    Leave[i] = df_trips['Leave'][i]
    Arrive[i] = df_trips['Arrive'][i]

In [3]:
# visits are a bit more challenging: I want this to be a set (or a list) for each trip
Visits = {}
for i in df_trips.index:
    tmpList = []
    # each trip has at least two visits
    tmpList.append(df_trips['Visit 1'][i])
    tmpList.append(df_trips['Visit 2'][i])
    # and optionally a third visit (check if not not null)
    if pd.notnull(df_trips['Visit 3'][i]):
        # note: because of the missing values (NaN), this column is interpreted as type "float".
        # to make it an integer element, I cast it as "int"        
        tmpList.append(int(df_trips['Visit 3'][i]))
    Visits[i] = tmpList

In [4]:
# for convenience, define two more index sets
Deliveries = df_deliveries.index
Trips = df_trips.index

In [5]:
# Define delivery/trip compatibility
#
# We are making some strong assumptions here: 
# - all deliveries are from 1 to 4, from 1 to 5, from 2 to 4, or from 2 to 5;
# - trips are either direct or via hub denoted by 3 (all starting from {1,2} and ending in {4,5})
# Therefore, we can simply test whether the origin and destination are part of the visits of each trip, and
# whether the Available/Leave times and LatestArrival/Arrive times are compatible.
#
# In general, we would need to have access to the direction of the trip, as well as the intermediate
# times of the visits to compile this informatio (as is done in the USPS paper).
CanAssign = {}
for i in Deliveries:
    for r in Trips:
        # (note: we can use a backslash \ to continue a statement on the next line)
        if Origin[i] in Visits[r] and Destination[i] in Visits[r] \
        and Available[i] <= Leave[r] and Arrive[r] <= LatestArrival[i]:
            CanAssign[i,r] = 1

In [6]:
from docplex.mp.model import Model
mdl = Model()

In [7]:
# variables
SelectTrip = mdl.binary_var_dict(Trips, name='select trip')

# save on memory by only defining variables that can be assigned:
# use the keys from 'CanAssign' here:
Proportion = mdl.continuous_var_dict(CanAssign.keys(), lb=0, ub=1, name='allocate proportion')

In [8]:
# objective
mdl.minimize(mdl.sum(Cost[r]*SelectTrip[r] for r in Trips))

In [9]:
# Constraints: allocate each delivery
for i in Deliveries:
    mdl.add_constraint(mdl.sum(Proportion[i,r] for r in Trips if (i,r) in CanAssign.keys()) == 1)

In [10]:
# Constraints: meet route capacity / linkage
for r in Trips:
    mdl.add_constraint(mdl.sum(Volume[i]*Proportion[i,r] for i in Deliveries if (i,r) in CanAssign.keys()) 
                       <= Capacity[r]*SelectTrip[r])

In [11]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.047,status='integer optimal solution')

In [12]:
mdl.print_solution()

objective: 11150
  "select trip_4"=1
  "select trip_5"=1
  "select trip_6"=1
  "select trip_7"=1
  "select trip_8"=1
  "select trip_9"=1
  "select trip_10"=1
  "select trip_12"=1
  "select trip_13"=1
  "select trip_15"=1
  "select trip_22"=1
  "select trip_25"=1
  "allocate proportion_d1_13"=0.219
  "allocate proportion_d1_25"=0.781
  "allocate proportion_d2_13"=0.393
  "allocate proportion_d2_15"=0.607
  "allocate proportion_d3_15"=1.000
  "allocate proportion_d4_4"=1.000
  "allocate proportion_d5_5"=1.000
  "allocate proportion_d6_6"=1.000
  "allocate proportion_d7_9"=1.000
  "allocate proportion_d8_10"=1.000
  "allocate proportion_d9_22"=1.000
  "allocate proportion_d10_7"=1.000
  "allocate proportion_d11_8"=1.000
  "allocate proportion_d12_12"=1.000


In [13]:
for r in Trips:
    if SelectTrip[r].solution_value >= 0.99:
        print("Trip " + str(r) + " has allocated volume " +
              str(mdl.sum(Volume[i]*Proportion[i,r] for i in Deliveries if (i,r) in CanAssign).solution_value) +
             " and capacity " + str(Capacity[r]))

Trip 4 has allocated volume 2500.0 and capacity 2500
Trip 5 has allocated volume 2300.0 and capacity 3000
Trip 6 has allocated volume 1700.0 and capacity 3000
Trip 7 has allocated volume 2000.0 and capacity 2000
Trip 8 has allocated volume 2100.0 and capacity 3000
Trip 9 has allocated volume 1650.0 and capacity 3000
Trip 10 has allocated volume 1800.0 and capacity 2500
Trip 12 has allocated volume 1340.0 and capacity 3000
Trip 13 has allocated volume 1799.9999999999995 and capacity 2500
Trip 15 has allocated volume 3000.0000000000005 and capacity 3000
Trip 22 has allocated volume 1300.0 and capacity 2500
Trip 25 has allocated volume 2500.0 and capacity 2500
